In [273]:
import numpy as np
import scipy.sparse as sp
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from scipy.sparse import spdiags
import imageio
import time
import random

In [274]:
#Creating the finite difference operators and defining constants
#Defining dimmensions
m = 128
n = m*m
nu = 0.001
L = 20

#Defining the domain
x1 = np.linspace(-L, L, m+1)
x = x1[:-1]
y = x
d = x[1]-x[0]
X, Y = np.meshgrid(x, y)

# Constructing the matrix A --> Laplacian operator
a0 = np.zeros((n, 1))
a1 = np.ones((n, 1))
a2 = np.ones((n, 1))    
a4 = np.zeros((n, 1)) 
for j in range(1, m+1):
    a2[m*j-1] = 0 
    a4[m*j-1] = 1  
a3 = np.roll(a2, shift=1)
a5 = np.roll(a4, shift =1)
a_diagonals = [a1.flatten(), a1.flatten(), a5.flatten(), 
             a2.flatten(), -4 * a1.flatten(), a3.flatten(), 
             a4.flatten(), a1.flatten(), a1.flatten()]
a_offsets = np.array([-(n-m), -m, -(m-1), -1, 0, 1, (m-1), m, (n-m)])
A = spdiags(a_diagonals, a_offsets, n, n).tolil() 
A[0,0] = 2     #From the note on the HW brief
A = A.tocsc()*(1/(d**2))

# Constructing the matrix B --> derivative operator for the first derivative with respect to x
b1 = np.ones((n,1)).flatten()
b2 = np.ones((n,1)).flatten()*(-1)
B_diagonals = [b1, b2, b1, b2]
B_offsets = np.array([-(n-m), -(m), (m), (n-m)])
B = spdiags(B_diagonals, B_offsets, n, n)
B = B.tocsc()*(1/(2*d))

#Constructing the matrix C --> derivative operator for the first derivative with respect to y
c1 = np.zeros((n,1))
c3 = np.ones((n,1))
for j in range(1, m+1):
    c3[m*j-1] = 0 
    c1[m*j-1] = 1    
c2 = c3.flatten()*(-1)
c4 = c1.flatten()*(-1)
c3 = np.roll(c3, shift=1).flatten()
c1 = np.roll(c1, shift=1).flatten()
C_diagonals = [c1, c2, c3, c4]
C_offsets = np.array([-(m-1), -1, 1, (m-1)])
C = spdiags(C_diagonals, C_offsets, n, n)
C = C.tocsc()*(1/(2*d))

In [275]:
#Part A: Creating each different initial condition and the function to solve

#Defining the time interval
t_span = [0, 60]
dt = 0.5
t_eval = np.arange(t_span[0], t_span[-1] + .2 , dt)

#Solving the vorticity equation using fft to solve for psi
Lk = 20
kx = (2 * np.pi / Lk) * np.concatenate((np.arange(0, m//2), np.arange(-m//2, 0)))
kx[0] = 1e-6
ky = kx
KX, KY = np.meshgrid(kx, ky)
K = KX**2 + KY**2

def wt_fourier(t, w, A, B, C, nu, K):
    w_2d = w.reshape((m,m), order='F')
    wf = np.fft.fft2(w_2d)
    psi_ft = (-1*wf)/(K)
    psi_f = np.fft.ifft2(psi_ft).real
    psi_f = psi_f.reshape(m**2,order='F')
    wt_f = (-B@psi_f)*(C@w) + (C@psi_f)*(B@w) + (nu)*(A@w)
    return wt_f

#Defining initial conditions
init_condits = []

#Initial condition 1: two oppositely “charged” Gaussian vortices next to each other
w_init1 = 20*np.exp(-((X+1)**2)/40 -(Y+2)**2/40) - 20*np.exp(-((X-1)**2)/40-(Y-2)**2/40)
w0_1 = w_init1.reshape(m**2,order='F')
init_condits.append(w0_1)

#Initial condition 2: Two same “charged” Gaussian vortices next to each other.
w_init2 = 3*np.exp(-(X+8)**2/40-(Y**2)/40) + 3*np.exp(-(X-8)**2/40-(Y**2)/40)
w0_2 = w_init2.reshape(m**2,order='F')
init_condits.append(w0_2)

#Initial condition 3: Two pairs of oppositely “charged” vortices which can be made to collide with each other.
x3 = 5
y3 = 5
mag = 8
w_init3 = mag*np.exp(-((X+x3)**2)/40 -(Y+y3)**2/40) - mag*np.exp(-((X-x3)**2)/40 -(Y+y3)**2/40) - mag*np.exp(-((X+x3)**2)/40 -(Y-y3)**2/40) + mag*np.exp(-((X-x3)**2)/40 -(Y-y3)**2/40)
w0_3 = w_init3.reshape(m**2,order='F')
init_condits.append(w0_3)

#Initial condition 4: A random assortment (in position, strength, charge, ellipticity, etc.) of vortices on the periodic domain
def random_vortex(number, max_strength, max_ellip, x, y):
    a, b = number
    vortex_num = random.randint(a, b)
    metrics = {'num':vortex_num, 'coords':[], 'mag': [], 'ellip':[] }
    w_init4 = np.zeros((m,m))
    for bumps in range(vortex_num):
        x_coord = random.uniform(x[0], x[-1]); y_coord = random.uniform(y[0], y[-1])
        strength = random.uniform(-max_strength, max_strength)
        ellip_x = random.uniform(0, max_ellip); ellip_y = random.uniform(0, max_ellip)
        metrics['coords'].append((x_coord, y_coord)); metrics['mag'].append(strength); metrics['ellip'].append((ellip_x, ellip_y))
        vortex = strength*np.exp(-((X+x_coord)**2/((ellip_x)))-((Y+y_coord)**2/((ellip_y))))
        w_init4 += vortex
        w0_4 = w_init4.reshape(m**2,order='F')
    return w0_4, metrics

w0_4, metrics = random_vortex((10, 15), 5, 40, x, y)
init_condits.append(w0_4)


In [272]:
#Part B: animating each solution for each initial condition
for k in range(len(init_condits)):
    init = init_condits[k]
    sol = solve_ivp(wt_fourier, t_span, init, t_eval=t_eval, args=(A, B, C, nu, K))
    frames = []
    
    for j in range(len(sol.t)):
        curw = sol.y[:,j].reshape((m,m), order= 'F')
        plt.figure(figsize=(5,4))
        plt.pcolormesh(X, Y, curw, cmap = 'viridis', shading='gouraud')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig("frame.png", dpi=100)
        plt.close()
        frames.append(imageio.v2.imread("frame.png"))
        
    imageio.mimsave(f"solution{k+1}.gif", frames, duration=0.05)